# Baseline Machine Learning for Store Sales Forecasting

This notebook builds a strong baseline for the Kaggle *Store Sales - Time Series Forecasting* dataset.

What this notebook covers:
- load and merge the raw retail datasets
- create leakage-safe features
- build a chronological train/validation/test split
- train simple baseline regressors
- evaluate the models with forecasting metrics

The goal is not to over-engineer the first model. The goal is to create a stable, explainable baseline that can be improved later with LSTM, Transformer, or more advanced gradient-boosted models.

## 1. Setup and imports

This section loads the standard scientific Python stack and points the notebook to the project root so the `src` package can be imported cleanly.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.preprocess import SplitConfig, prepare_datasets, split_by_time
from src.data.features import FeatureConfig, build_features

warnings_flag = False
try:
    import warnings
    warnings.filterwarnings('ignore')
    warnings_flag = True
except Exception:
    pass

pd.set_option('display.max_columns', 200)
sns.set_theme(style='whitegrid')
print(f'Project root: {ROOT}')
print(f'Warnings ignored: {warnings_flag}')

Project root: C:\Users\MONTAHA\Desktop\Ai\forecasting-ml-system
Warnings ignored: True


## 2. Load and preprocess the retail panel

The preprocessing module handles raw loading, safe merges, holiday expansion, oil alignment, transaction cleaning, and train-based outlier capping.

Important design choice: the preprocessing is fit on training data only, so the validation and test sets do not leak future information into the cleaning step.

In [ ]:
data_dir = ROOT / 'data' / 'raw'
split_config = SplitConfig(validation_days=30, test_days=16, date_col='date')

datasets = prepare_datasets(data_dir=data_dir, split_config=split_config)
full_data = datasets['full']
train_data = datasets['train']
validation_data = datasets['validation']
test_data = datasets['test']

print('Dataset sizes:')
print(f"full      -> {full_data.shape}")
print(f"train     -> {train_data.shape}")
print(f"validation-> {validation_data.shape}")
print(f"test      -> {test_data.shape}")

display(full_data.head())

Dataset sizes:
full      -> (3000888, 20)
train     -> (2918916, 20)
validation-> (53460, 20)
test      -> (28512, 20)


,id,date,store_nbr,family,sales,onpromotion,city,state,type,cluster,transactions,dcoilwtico,holiday_count,is_holiday,is_transfer_holiday,is_bridge_holiday,is_working_day,is_national_holiday,is_local_holiday,is_regional_holiday
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0.0,Quito,Pichincha,D,13,1746.5,53.95,1.0,1,0,0,0,1,0,0
1,1,2013-01-01,1,BABY CARE,0.0,0.0,Quito,Pichincha,D,13,1746.5,53.95,1.0,1,0,0,0,1,0,0
2,2,2013-01-01,1,BEAUTY,0.0,0.0,Quito,Pichincha,D,13,1746.5,53.95,1.0,1,0,0,0,1,0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0.0,Quito,Pichincha,D,13,1746.5,53.95,1.0,1,0,0,0,1,0,0
4,4,2013-01-01,1,BOOKS,0.0,0.0,Quito,Pichincha,D,13,1746.5,53.95,1.0,1,0,0,0,1,0,0


## 3. Create leakage-safe features

We now transform the merged panel into model-ready features.

The key rule is that every historical signal used by the model must come from the past only. That means lag features and rolling statistics are shifted before aggregation.

In [ ]:
feature_config = FeatureConfig(
    date_col='date',
    group_cols=('store_nbr', 'family'),
    target_col='sales',
    transaction_col='transactions',
    promotion_col='onpromotion',
    oil_col='dcoilwtico',
    lags=(1, 7, 14, 28),
    rolling_windows=(7, 14, 28),
    drop_incomplete_rows=True,
)

featured = build_features(full_data, config=feature_config)
print(f'Featured frame shape: {featured.shape}')
display(featured.head())

Featured frame shape: (2950992, 102)


,id,date,store_nbr,family,sales,onpromotion,city,state,type,cluster,transactions,dcoilwtico,holiday_count,is_holiday,is_transfer_holiday,is_bridge_holiday,is_working_day,is_national_holiday,is_local_holiday,is_regional_holiday,day,dayofweek,weekofyear,month,quarter,year,dayofyear,is_weekend,is_month_start,is_month_end,dayofweek_sin,dayofweek_cos,month_sin,month_cos,dayofyear_sin,dayofyear_cos,promo_active,sales_lag_1,sales_lag_7,sales_lag_14,sales_lag_28,sales_roll_mean_7,sales_roll_std_7,sales_roll_min_7,sales_roll_max_7,sales_roll_mean_14,sales_roll_std_14,sales_roll_min_14,sales_roll_max_14,sales_roll_mean_28,sales_roll_std_28,sales_roll_min_28,sales_roll_max_28,transactions_lag_1,transactions_lag_7,transactions_lag_14,transactions_lag_28,transactions_roll_mean_7,transactions_roll_std_7,transactions_roll_min_7,transactions_roll_max_7,transactions_roll_mean_14,transactions_roll_std_14,transactions_roll_min_14,transactions_roll_max_14,transactions_roll_mean_28,transactions_roll_std_28,transactions_roll_min_28,transactions_roll_max_28,promo_lag_1,promo_lag_7,promo_lag_14,promo_lag_28,promo_roll_mean_7,promo_roll_std_7,promo_roll_min_7,promo_roll_max_7,promo_roll_mean_14,promo_roll_std_14,promo_roll_min_14,promo_roll_max_14,promo_roll_mean_28,promo_roll_std_28,promo_roll_min_28,promo_roll_max_28,oil_lag_1,oil_lag_7,oil_lag_14,oil_lag_28,oil_roll_mean_7,oil_roll_std_7,oil_roll_min_7,oil_roll_max_7,oil_roll_mean_14,oil_roll_std_14,oil_roll_min_14,oil_roll_max_14,oil_roll_mean_28,oil_roll_std_28,oil_roll_min_28,oil_roll_max_28,days_since_start
0,49896,2013-01-29,1,AUTOMOTIVE,2.0,0.0,Quito,Pichincha,D,13,1772.0,97.62,0,0,0,0,0,0,0,0,29,1,5,1,1,2013,29,0,0,0,0.781831,0.62349,0.0,1.0,0.463258,0.886224,0,3.0,1.0,1.0,0.0,2.571429,1.718249,0.0,5.0,2.142857,1.703261,0.0,5.0,2.142857,1.458418,0.0,5.0,1738.0,1762.0,1680.0,1746.5,1513.714286,467.126933,542.0,1873.0,1529.0,458.405934,507.0,1933.0,1566.482143,462.778274,507.0,2111.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,95.95,96.09,93.26,53.95,95.414286,0.424808,95.06,96.09,95.240714,0.720443,93.26,96.09,92.898571,7.710398,53.95,96.09,28
1,49897,2013-01-29,1,BABY CARE,0.0,0.0,Quito,Pichincha,D,13,1772.0,97.62,0,0,0,0,0,0,0,0,29,1,5,1,1,2013,29,0,0,0,0.781831,0.62349,0.0,1.0,0.463258,0.886224,0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,1738.0,1762.0,1680.0,1746.5,1513.714286,467.126933,542.0,1873.0,1529.0,458.405934,507.0,1933.0,1566.482143,462.778274,507.0,2111.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,95.95,96.09,93.26,53.95,95.414286,0.424808,95.06,96.09,95.240714,0.720443,93.26,96.09,92.898571,7.710398,53.95,96.09,28
2,49898,2013-01-29,1,BEAUTY,3.0,0.0,Quito,Pichincha,D,13,1772.0,97.62,0,0,0,0,0,0,0,0,29,1,5,1,1,2013,29,0,0,0,0.781831,0.62349,0.0,1.0,0.463258,0.886224,0,0.0,1.0,0.0,0.0,0.571429,0.786796,0.0,2.0,1.214286,1.423893,0.0,4.0,1.321429,1.334821,0.0,4.0,1738.0,1762.0,1680.0,1746.5,1513.714286,467.126933,542.0,1873.0,1529.0,458.405934,507.0,1933.0,1566.482143,462.778274,507.0,2111.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,95.95,96.09,93.26,53.95,95.414286,0.424808,95.06,96.09,95.240714,0.720443,93.26,96.09,92.898571,7.710398,53.95,96.09,28
3,49899,2013-01-29,1,BEVERAGES,932.0,0.0,Quito,Pichincha,D,13,1772.0,97.62,0,0,0,0,0,0,0,0,29,1,5,1,1,2013,29,0,0,0,0.781831,0.62349,0.0,1.0,0.463258,0.886224,0,772.0,1037.0,1149.0,0.0,884.571429,237.427082,417.0,1103.0,946.357143,252.828494,417.0,1280.0,925.892857,300.124584,0.0,1280.0,1738.0,1762.0,1680.0,1746.5,1513.714286,467.126933,542.0,1873.0,1529.0,458.405934,507.0,1933.0,1566.482143,462.778274,507.0,2111.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,95.95,96.09,93.26,53.95,95.414286,0.424808,95.06,96.09,95.240714,0.720443,93.26,96.09,92.898571,7.710398,53.95,96.09,28
4,49900,2013-01-29,1,BOOKS,0.0,0.0,Quito,Pichincha,D,13,1772.0,97.62,0,0,0,0,0,0,0,0,29,1,5,1,1,2013,29,0,0,0,0.781831,0.62349,0.

### Notes on the feature design

- `sales_lag_*` captures short and medium-term demand memory.
- rolling mean/std/min/max helps the model understand local trend and volatility.
- calendar features capture weekly and monthly seasonality.
- promotion, transaction, and oil features are kept only as past-based signals.

This is a good baseline structure because it is simple, robust, and explainable.

In [ ]:
target_col = 'sales'
date_col = 'date'

# Use unique dates (featured has many rows per date)
unique_dates = featured[date_col].drop_duplicates().sort_values().to_numpy()
validation_start = unique_dates[-46]  # 30 validation + 16 test days from the end
test_start = unique_dates[-16]

train_df = featured[featured[date_col] < validation_start].copy()
val_df = featured[(featured[date_col] >= validation_start) & (featured[date_col] < test_start)].copy()
test_df = featured[featured[date_col] >= test_start].copy()

print('Feature split sizes:')
print(f"train -> {train_df.shape}")
print(f"val   -> {val_df.shape}")
print(f"test  -> {test_df.shape}")

print('Date ranges:')
print(f"train: {train_df[date_col].min()} -> {train_df[date_col].max()}")
print(f"val  : {val_df[date_col].min()} -> {val_df[date_col].max()}")
print(f"test : {test_df[date_col].min()} -> {test_df[date_col].max()}")

Feature split sizes:
train -> (2869020, 102)
val   -> (53460, 102)
test  -> (28512, 102)
Date ranges:
train: 2013-01-29 00:00:00 -> 2017-06-30 00:00:00
val  : 2017-07-01 00:00:00 -> 2017-07-30 00:00:00
test : 2017-07-31 00:00:00 -> 2017-08-15 00:00:00


## 4. Define baseline target and features

For a first baseline, we predict daily sales directly. The feature set includes time-derived variables, lagged demand, rolling statistics, promotions, transactions, oil, and categorical store/family identifiers.

In [ ]:
drop_columns = [
    'sales',
    'date',
]

X_train = train_df.drop(columns=[c for c in drop_columns if c in train_df.columns])
y_train = train_df[target_col].astype(float)

X_val = val_df.drop(columns=[c for c in drop_columns if c in val_df.columns])
y_val = val_df[target_col].astype(float)

X_test = test_df.drop(columns=[c for c in drop_columns if c in test_df.columns])
y_test = test_df[target_col].astype(float)

numeric_features = X_train.select_dtypes(include=['number', 'bool']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

print(f'Numeric features: {len(numeric_features)}')
print(f'Categorical features: {len(categorical_features)}')
print('Categorical sample:', categorical_features[:10])

Numeric features: 96
Categorical features: 4
Categorical sample: ['family', 'city', 'state', 'type']


## 5. Model 1: naive seasonal baseline

Before training any machine learning model, we evaluate a simple naive baseline.

Here the forecast is the 7-day lagged sales value. This is a strong sanity check because time-series models should beat it clearly to be considered useful.

In [ ]:
def rmsle(y_true, y_pred):
    y_true = np.maximum(np.asarray(y_true), 0)
    y_pred = np.maximum(np.asarray(y_pred), 0)
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2))

naive_col = 'sales_lag_7'
naive_val = val_df[naive_col].fillna(train_df[target_col].mean())
naive_test = test_df[naive_col].fillna(train_df[target_col].mean())

naive_metrics = {
    'validation_mae': mean_absolute_error(y_val, naive_val),
    'validation_rmse': np.sqrt(mean_squared_error(y_val, naive_val)),
    'validation_rmsle': rmsle(y_val, naive_val),
    'test_mae': mean_absolute_error(y_test, naive_test),
    'test_rmse': np.sqrt(mean_squared_error(y_test, naive_test)),
    'test_rmsle': rmsle(y_test, naive_test),
}

pd.Series(naive_metrics).to_frame('value')

,value
validation_mae,78.046072
validation_rmse,284.345852
validation_rmsle,0.507114
test_mae,97.127714
test_rmse,335.526838
test_rmsle,0.550792


## 6. Model 2: ridge regression baseline

A regularized linear model is a good baseline for retail forecasting because it is fast, stable, and easy to interpret.

We use one-hot encoding for categorical variables and standardization for numeric variables. Missing numeric values are imputed with the median, and missing categorical values with the most frequent category.

In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ],
    remainder='drop',
)

ridge_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', Ridge(alpha=1.0, random_state=42)),
])

ridge_model.fit(X_train, y_train)
ridge_val_pred = ridge_model.predict(X_val)
ridge_test_pred = ridge_model.predict(X_test)

ridge_metrics = {
    'validation_mae': mean_absolute_error(y_val, ridge_val_pred),
    'validation_rmse': np.sqrt(mean_squared_error(y_val, ridge_val_pred)),
    'validation_rmsle': rmsle(y_val, ridge_val_pred),
    'test_mae': mean_absolute_error(y_test, ridge_test_pred),
    'test_rmse': np.sqrt(mean_squared_error(y_test, ridge_test_pred)),
    'test_rmsle': rmsle(y_test, ridge_test_pred),
}

pd.Series(ridge_metrics).to_frame('value')

,value
validation_mae,70.036878
validation_rmse,203.543376
validation_rmsle,1.308260
test_mae,81.998104
test_rmse,217.591755
test_rmsle,1.412463


### Baseline comparison

This table compares the naive seasonal forecast with the ridge regression baseline. In practice, the ridge model should outperform the naive method if the feature engineering is effective.

In [ ]:
comparison = pd.DataFrame({
    'naive_validation_rmsle': [naive_metrics['validation_rmsle']],
    'ridge_validation_rmsle': [ridge_metrics['validation_rmsle']],
    'naive_test_rmsle': [naive_metrics['test_rmsle']],
    'ridge_test_rmsle': [ridge_metrics['test_rmsle']],
})
comparison.T.rename(columns={0: 'score'})

,score
naive_validation_rmsle,0.507114
ridge_validation_rmsle,1.308260
naive_test_rmsle,0.550792
ridge_test_rmsle,1.412463


## 7. Model 3: random forest baseline

Tree-based models can capture nonlinear interactions between promotions, seasonality, and lag features.

This model is usually heavier than ridge regression, but it can be useful as a stronger classical benchmark.

In [ ]:
rf_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=120,
        max_depth=18,
        min_samples_leaf=5,
        n_jobs=-1,
        random_state=42,
    )),
])

rf_model.fit(X_train, y_train)
rf_val_pred = rf_model.predict(X_val)
rf_test_pred = rf_model.predict(X_test)

rf_metrics = {
    'validation_mae': mean_absolute_error(y_val, rf_val_pred),
    'validation_rmse': mean_squared_error(y_val, rf_val_pred, squared=False),
    'validation_rmsle': rmsle(y_val, rf_val_pred),
    'test_mae': mean_absolute_error(y_test, rf_test_pred),
    'test_rmse': mean_squared_error(y_test, rf_test_pred, squared=False),
    'test_rmsle': rmsle(y_test, rf_test_pred),
}

pd.Series(rf_metrics).to_frame('value')

## 8. Final interpretation

Use the best-performing validation model for any final report, then re-evaluate once on the test set.

For the Kaggle project, the next step after this notebook is usually to move from classical baselines to a sequence model such as LSTM or Transformer, keeping the same leakage-safe data pipeline.

In [ ]:
results = pd.DataFrame([
    {'model': 'naive_seasonal', 'validation_rmsle': naive_metrics['validation_rmsle'], 'test_rmsle': naive_metrics['test_rmsle']},
    {'model': 'ridge_regression', 'validation_rmsle': ridge_metrics['validation_rmsle'], 'test_rmsle': ridge_metrics['test_rmsle']},
    {'model': 'random_forest', 'validation_rmsle': rf_metrics['validation_rmsle'], 'test_rmsle': rf_metrics['test_rmsle']},
])
results.sort_values('validation_rmsle')